In [ ]:
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter
import os

OUTPUT_DIR = os.path.join(os.getcwd(), "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
COLUMNS = [
    # (Column Name, Data Type, Example Value, Business Description, Constraints / Notes)
    (
        "referral_details_id",
        "Integer",
        "1",
        "A unique sequential number assigned to each row in this report. "
        "Use this to reference a specific record.",
        "Auto-generated. Never null. Starts at 1.",
    ),
    (
        "referral_id",
        "Text (ID)",
        "9331c8f144dad5a3b8e4a10467b4343a",
        "The unique identifier for the referral event. "
        "Links back to the referral program database.",
        "Never null. 32-character alphanumeric code.",
    ),
    (
        "referral_source",
        "Text",
        "Draft Transaction",
        "How the referral was created. "
        "Possible values: 'User Sign Up' (referee registered online), "
        "'Draft Transaction' (referee made an offline purchase), "
        "'Lead' (referee was a tracked sales lead).",
        "Never null. One of three values.",
    ),
    (
        "referral_source_category",
        "Text",
        "Offline",
        "A simplified category of the referral channel. "
        "'Online' = User Sign Up. "
        "'Offline' = Draft Transaction. "
        "For Lead referrals, the category comes from the lead tracking system "
        "(e.g. 'Online', 'Offline').",
        "Never null. Derived from referral_source.",
    ),
    (
        "referral_at",
        "Datetime",
        "2024-05-01 12:17:31",
        "The date and time when the referral was first recorded "
        "in the system. Shown in the referrer's local time.",
        "Format: YYYY-MM-DD HH:MM:SS. May be blank if timezone info is missing.",
    ),
    (
        "referrer_id",
        "Text (ID)",
        "2c71c5d66c7e12a0b3c200ba6ed3b78e",
        "The unique identifier of the existing member who made the referral "
        "(the person who invited someone new).",
        "Hashed for privacy. May be blank if referrer data is unavailable.",
    ),
    (
        "referrer_name",
        "Text",
        "4a9f2b1c8e3d7055f6a0b4c2e9d1f8a3",
        "The name of the referrer. "
        "Note: stored in hashed format to protect member privacy.",
        "Hashed for PII protection. May be blank.",
    ),
    (
        "referrer_phone_number",
        "Text",
        "87e6571bf783832fffc616a308563e7e",
        "The phone number of the referrer. "
        "Stored in hashed format to protect member privacy.",
        "Hashed for PII protection. May be blank.",
    ),
    (
        "referrer_homeclub",
        "Text",
        "BENHIL",
        "The home club (gym location) of the referrer. "
        "Shown in uppercase as the official club code.",
        "Club codes are not title-cased. May be blank.",
    ),
    (
        "referee_id",
        "Text (ID)",
        "f1327c9d6d4efee6ad69e7e467b605b9",
        "The unique identifier of the new member who was referred. "
        "Only populated when the referral source is 'Lead'.",
        "Blank for 'User Sign Up' and 'Draft Transaction' sources.",
    ),
    (
        "referee_name",
        "Text",
        "2e68f7f5c8854bd2cf2b5ff55bc7e780",
        "The name of the new member (referee). "
        "Stored in hashed format to protect member privacy.",
        "Hashed for PII protection. May be blank.",
    ),
    (
        "referee_phone",
        "Text",
        "321b0102e766ef73b63d1bd797203c02",
        "The phone number of the new member (referee). "
        "Stored in hashed format to protect member privacy.",
        "Hashed for PII protection. May be blank.",
    ),
    (
        "referral_status",
        "Text",
        "Berhasil",
        "The current outcome of the referral. "
        "'Berhasil' = Successful (reward may be granted). "
        "'Menunggu' = Pending (waiting for transaction to complete). "
        "'Tidak Berhasil' = Failed (referral did not convert).",
        "One of three values. Never null.",
    ),
    (
        "num_reward_days",
        "Integer",
        "10",
        "The number of free membership days awarded to the referrer "
        "if the referral is successful. "
        "0 means no reward is assigned to this referral.",
        "0 when referral_reward_id is null or reward value is zero.",
    ),
    (
        "transaction_id",
        "Text (ID)",
        "e05121ea99fed4f4c5224a4667bb3dad",
        "The unique identifier of the payment transaction linked to this referral. "
        "This is the transaction made by the new member (referee).",
        "Blank when no transaction is linked (e.g. pending referrals).",
    ),
    (
        "transaction_status",
        "Text",
        "Paid",
        "The payment status of the linked transaction. "
        "'Paid' = payment was successfully completed.",
        "Blank when no transaction is linked.",
    ),
    (
        "transaction_at",
        "Datetime",
        "2024-05-14 13:17:03",
        "The date and time when the linked payment transaction occurred. "
        "Shown in the transaction location's local time.",
        "Format: YYYY-MM-DD HH:MM:SS. Blank when no transaction exists.",
    ),
    (
        "transaction_location",
        "Text",
        "ADITYAWARMAN",
        "The club location where the transaction took place. "
        "Shown in uppercase as the official location code.",
        "Blank when no transaction is linked.",
    ),
    (
        "transaction_type",
        "Text",
        "New",
        "The type of membership purchased in the linked transaction. "
        "'New' = brand-new membership. "
        "'Rejoin' = returning member rejoining.",
        "Blank when no transaction is linked. "
        "Only 'New' transactions qualify for referral rewards.",
    ),
    (
        "updated_at",
        "Datetime",
        "2024-05-01 12:17:31",
        "The date and time when this referral record was last updated "
        "in the system. Shown in the referrer's local time.",
        "Format: YYYY-MM-DD HH:MM:SS.",
    ),
    (
        "reward_granted_at",
        "Datetime",
        "2024-05-14 13:17:36",
        "The date and time when the reward (free membership days) was "
        "officially granted to the referrer's account. "
        "Blank if the reward has not yet been disbursed.",
        "Format: YYYY-MM-DD HH:MM:SS. Blank when reward not yet granted.",
    ),
    (
        "is_business_logic_valid",
        "Boolean (True/False)",
        "True",
        "Indicates whether this referral record passes all business rules. "
        "TRUE = the referral is legitimate and reward (if any) is correctly assigned. "
        "FALSE = a rule violation was detected — this record may indicate fraud, "
        "a data error, or a reward assigned incorrectly.",
        "Key fraud-detection flag. See 'Business Rules' sheet for full logic.",
    ),
]

In [ ]:
VALID_CONDITIONS = [
    ("Valid — Condition A", "Successful referral",
     "Status is 'Berhasil' AND reward > 0 AND has transaction AND "
     "transaction status = PAID AND transaction type = NEW AND "
     "transaction occurred AFTER referral AND in the SAME MONTH AND "
     "referrer membership has not expired AND referrer account is active"),
    ("Valid — Condition B", "Pending or failed referral",
     "Status is 'Menunggu' or 'Tidak Berhasil' AND no reward assigned (reward = 0)"),
]

INVALID_CONDITIONS = [
    ("Invalid — Rule 1", "Reward on wrong status",
     "Reward value > 0 AND status is NOT 'Berhasil'"),
    ("Invalid — Rule 2", "Reward without transaction",
     "Reward value > 0 AND no transaction ID linked to the referral"),
    ("Invalid — Rule 3", "Paid transaction with no reward",
     "No reward assigned AND has transaction AND transaction status = PAID AND "
     "transaction occurred after referral"),
    ("Invalid — Rule 4", "Successful with no reward",
     "Status is 'Berhasil' AND reward is null or zero"),
    ("Invalid — Rule 5", "Transaction before referral",
     "The linked transaction occurred BEFORE the referral was created "
     "(time-travel fraud signal)"),
]


In [ ]:
DARK_BLUE   = "1F3864"
MID_BLUE    = "2E75B6"
LIGHT_BLUE  = "BDD7EE"
LIGHT_GRAY  = "F2F2F2"
WHITE       = "FFFFFF"
VALID_GREEN = "E2EFDA"
INVALID_RED = "FCE4D6"
HEADER_FONT = Font(name="Arial", bold=True, color=WHITE, size=11)
BODY_FONT   = Font(name="Arial", size=10)
BOLD_FONT   = Font(name="Arial", bold=True, size=10)
WRAP        = Alignment(wrap_text=True, vertical="top")
CENTER      = Alignment(horizontal="center", vertical="top", wrap_text=True)

def header_fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)

def thin_border():
    s = Side(style="thin", color="BFBFBF")
    return Border(left=s, right=s, top=s, bottom=s)

def apply_border(cell):
    cell.border = thin_border()

In [ ]:
wb = Workbook()

In [ ]:
ws = wb.active
ws.title = "Data Dictionary"

ws.sheet_view.showGridLines = False

# Title row
ws.merge_cells("A1:E1")
ws["A1"] = "Referral Report — Data Dictionary"
ws["A1"].font = Font(name="Arial", bold=True, color=WHITE, size=14)
ws["A1"].fill = header_fill(DARK_BLUE)
ws["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[1].height = 30

# Subtitle
ws.merge_cells("A2:E2")
ws["A2"] = "For: Marketing Managers & Business Users    |    Generated by: Referral Data Pipeline"
ws["A2"].font = Font(name="Arial", italic=True, color=DARK_BLUE, size=10)
ws["A2"].fill = header_fill(LIGHT_BLUE)
ws["A2"].alignment = Alignment(horizontal="center", vertical="center")
ws.row_dimensions[2].height = 18

# Blank
ws.row_dimensions[3].height = 6

# Column headers
headers = ["Column Name", "Data Type", "Example Value",
           "What It Means (Business Description)", "Constraints / Notes"]
for col_idx, h in enumerate(headers, start=1):
    cell = ws.cell(row=4, column=col_idx, value=h)
    cell.font = HEADER_FONT
    cell.fill = header_fill(MID_BLUE)
    cell.alignment = CENTER
    apply_border(cell)
ws.row_dimensions[4].height = 22

# Data rows
for row_idx, (col_name, dtype, example, desc, notes) in enumerate(COLUMNS, start=5):
    fill_color = LIGHT_GRAY if row_idx % 2 == 1 else WHITE
    row_fill = PatternFill("solid", fgColor=fill_color)

    values = [col_name, dtype, example, desc, notes]
    for col_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=val)
        cell.font = BOLD_FONT if col_idx == 1 else BODY_FONT
        cell.fill = row_fill
        cell.alignment = WRAP
        apply_border(cell)

    ws.row_dimensions[row_idx].height = 52

# Column widths
widths = [28, 20, 40, 65, 55]
for i, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = w

# Freeze header rows
ws.freeze_panes = "A5"



In [ ]:
ws2 = wb.create_sheet("Business Rules")
ws2.sheet_view.showGridLines = False

ws2.merge_cells("A1:C1")
ws2["A1"] = "is_business_logic_valid — Business Rules Reference"
ws2["A1"].font = Font(name="Arial", bold=True, color=WHITE, size=13)
ws2["A1"].fill = header_fill(DARK_BLUE)
ws2["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws2.row_dimensions[1].height = 28

ws2.merge_cells("A2:C2")
ws2["A2"] = (
    "This sheet explains the rules used to set is_business_logic_valid = TRUE or FALSE. "
    "FALSE rows are potential fraud or data anomalies that need investigation."
)
ws2["A2"].font = Font(name="Arial", italic=True, color=DARK_BLUE, size=10)
ws2["A2"].fill = header_fill(LIGHT_BLUE)
ws2["A2"].alignment = Alignment(wrap_text=True, horizontal="left", vertical="center")
ws2.row_dimensions[2].height = 28

row = 4

# Section: VALID
ws2.merge_cells(f"A{row}:C{row}")
ws2[f"A{row}"] = "✓  CONDITIONS THAT MAKE A REFERRAL VALID (is_business_logic_valid = TRUE)"
ws2[f"A{row}"].font = Font(name="Arial", bold=True, color=WHITE, size=11)
ws2[f"A{row}"].fill = header_fill("375623")
ws2[f"A{row}"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
ws2.row_dimensions[row].height = 22
row += 1

hdrs = ["Condition", "Type", "Full Rule Description"]
for ci, h in enumerate(hdrs, 1):
    c = ws2.cell(row=row, column=ci, value=h)
    c.font = HEADER_FONT
    c.fill = header_fill("548235")
    c.alignment = CENTER
    apply_border(c)
ws2.row_dimensions[row].height = 20
row += 1

for label, typ, desc in VALID_CONDITIONS:
    row_fill = PatternFill("solid", fgColor=VALID_GREEN)
    for ci, val in enumerate([label, typ, desc], 1):
        c = ws2.cell(row=row, column=ci, value=val)
        c.font = BODY_FONT
        c.fill = row_fill
        c.alignment = WRAP
        apply_border(c)
    ws2.row_dimensions[row].height = 46
    row += 1

row += 1

# Section: INVALID
ws2.merge_cells(f"A{row}:C{row}")
ws2[f"A{row}"] = "✗  CONDITIONS THAT MAKE A REFERRAL INVALID (is_business_logic_valid = FALSE)"
ws2[f"A{row}"].font = Font(name="Arial", bold=True, color=WHITE, size=11)
ws2[f"A{row}"].fill = header_fill("833C00")
ws2[f"A{row}"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
ws2.row_dimensions[row].height = 22
row += 1

for ci, h in enumerate(hdrs, 1):
    c = ws2.cell(row=row, column=ci, value=h)
    c.font = HEADER_FONT
    c.fill = header_fill(MID_BLUE)
    c.alignment = CENTER
    apply_border(c)
ws2.row_dimensions[row].height = 20
row += 1

for label, typ, desc in INVALID_CONDITIONS:
    row_fill = PatternFill("solid", fgColor=INVALID_RED)
    for ci, val in enumerate([label, typ, desc], 1):
        c = ws2.cell(row=row, column=ci, value=val)
        c.font = BODY_FONT
        c.fill = row_fill
        c.alignment = WRAP
        apply_border(c)
    ws2.row_dimensions[row].height = 46
    row += 1

ws2.column_dimensions["A"].width = 28
ws2.column_dimensions["B"].width = 30
ws2.column_dimensions["C"].width = 80
ws2.freeze_panes = "A5"


In [ ]:
ws3 = wb.create_sheet("Source Tables")
ws3.sheet_view.showGridLines = False

ws3.merge_cells("A1:D1")
ws3["A1"] = "Source Tables — Overview for Business Users"
ws3["A1"].font = Font(name="Arial", bold=True, color=WHITE, size=13)
ws3["A1"].fill = header_fill(DARK_BLUE)
ws3["A1"].alignment = Alignment(horizontal="center", vertical="center")
ws3.row_dimensions[1].height = 28

ws3.merge_cells("A2:D2")
ws3["A2"] = "The report is built by combining the 7 tables below. Each row in the report = one referral event."
ws3["A2"].font = Font(name="Arial", italic=True, color=DARK_BLUE, size=10)
ws3["A2"].fill = header_fill(LIGHT_BLUE)
ws3["A2"].alignment = Alignment(horizontal="left", vertical="center", indent=1)
ws3.row_dimensions[2].height = 20

SOURCE_TABLES = [
    ("user_referrals",         "46",  "Core table. One row per referral.",
     "referral_id, referral_at, referrer_id, referee_id, referral_source, transaction_id, status"),
    ("user_referral_logs",     "96",  "Audit log of referral state changes. Multiple rows per referral — collapsed to 1.",
     "user_referral_id, source_transaction_id, is_reward_granted, created_at"),
    ("user_logs",              "29",  "Member account details (names and phones are hashed for privacy).",
     "user_id, name, phone_number, homeclub, membership_expired_date, is_deleted"),
    ("lead_logs",              "8",   "Details of leads tracked by the sales team.",
     "lead_id, source_category, preferred_location, current_status"),
    ("user_referral_statuses", "3",   "Lookup: referral status codes.",
     "id: 1=Menunggu, 2=Berhasil, 3=Tidak Berhasil"),
    ("referral_rewards",       "3",   "Lookup: reward tiers (in days of free membership).",
     "id, reward_value (e.g. '10 days'), reward_type"),
    ("paid_transactions",      "14",  "Payment transactions that may be linked to referrals.",
     "transaction_id, transaction_status, transaction_at, transaction_type, transaction_location"),
]

row = 4
hdrs = ["Table Name", "Row Count", "What It Contains", "Key Columns"]
for ci, h in enumerate(hdrs, 1):
    c = ws3.cell(row=row, column=ci, value=h)
    c.font = HEADER_FONT
    c.fill = header_fill(MID_BLUE)
    c.alignment = CENTER
    apply_border(c)
ws3.row_dimensions[row].height = 20
row += 1

for i, (tbl, cnt, desc, keys) in enumerate(SOURCE_TABLES):
    fill_color = LIGHT_GRAY if i % 2 == 0 else WHITE
    row_fill = PatternFill("solid", fgColor=fill_color)
    for ci, val in enumerate([tbl, cnt, desc, keys], 1):
        c = ws3.cell(row=row, column=ci, value=val)
        c.font = BOLD_FONT if ci == 1 else BODY_FONT
        c.fill = row_fill
        c.alignment = WRAP
        apply_border(c)
    ws3.row_dimensions[row].height = 46
    row += 1

ws3.column_dimensions["A"].width = 28
ws3.column_dimensions["B"].width = 12
ws3.column_dimensions["C"].width = 45
ws3.column_dimensions["D"].width = 65
ws3.freeze_panes = "A5"


In [ ]:
out_path = os.path.join(OUTPUT_DIR, "data_dictionary.xlsx")
wb.save(out_path)
print(f"Saved: {out_path}")

Saved: /content/output/data_dictionary.xlsx


In [17]:
from google.colab import files
files.download(out_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>